# ЛР-4: ROS 2 MCAP без ROS

Среда: **Google Colab**.

Полное методическое описание, критерии оценивания и `TEACH CARD` находятся в одноимённом `.md` файле комплекта.

Сквозной пайплайн курса:

$$
\text{Sense} \rightarrow \text{Collect} \rightarrow \text{Stream} \rightarrow
\text{Store} \rightarrow \text{Process} \rightarrow \text{Learn} \rightarrow \text{Teach}
$$


In [ ]:
!pip -q install "mcap>=1.3,<2" "mcap-ros2-support>=0.5,<1" "rosbags==0.11.5" pandas pyarrow plotly

from pathlib import Path
import math
import pandas as pd
import plotly.express as px

from mcap_ros2.writer import Writer as McapWriter
from mcap_ros2.decoder import DecoderFactory
from mcap.reader import make_reader

OUT = Path("/content/robot_imu_ros2.mcap")

# Валидное ROS 2 message definition без установленного ROS 2.
SCHEMA_NAME = "std_msgs/Float64"
SCHEMA_TEXT = "float64 data"

# 1. Создание ROS 2 MCAP: CDR кодируется mcap-ros2-support.
with OUT.open("wb") as f:
    writer = McapWriter(f)
    schema = writer.register_msgdef(SCHEMA_NAME, SCHEMA_TEXT)

    frequency_hz = 50
    n = 1000
    t0_ns = 1_720_000_000_000_000_000

    for i in range(n):
        t = i / frequency_hz
        ts_ns = t0_ns + int(t * 1e9)

        ax = 0.20 * math.sin(2 * math.pi * 1.2 * t)
        ay = 0.12 * math.cos(2 * math.pi * 0.8 * t)
        az = 9.81 + 0.05 * math.sin(2 * math.pi * 2.0 * t)

        # Искусственная ударная аномалия.
        if 8.0 <= t < 8.3:
            ax += 3.0
            ay -= 2.0

        for topic, value in [
            ("/imu/ax", ax),
            ("/imu/ay", ay),
            ("/imu/az", az),
        ]:
            writer.write_message(
                topic=topic,
                schema=schema,
                message={"data": float(value)},
                log_time=ts_ns,
                publish_time=ts_ns,
                sequence=i,
            )

    writer.finish()

print("MCAP:", OUT, "size =", OUT.stat().st_size, "bytes")

# 2. Декодирование CDR без ROS 2.
rows = {}

with OUT.open("rb") as f:
    reader = make_reader(f, decoder_factories=[DecoderFactory()])
    for schema, channel, message, ros_msg in reader.iter_decoded_messages():
        ts = message.log_time
        rows.setdefault(ts, {"timestamp_ns": ts})
        rows[ts][channel.topic.split("/")[-1]] = float(ros_msg.data)

imu = pd.DataFrame(rows.values()).sort_values("timestamp_ns").reset_index(drop=True)
imu["time_s"] = (imu["timestamp_ns"] - imu["timestamp_ns"].min()) / 1e9
imu["accel_norm"] = (imu["ax"]**2 + imu["ay"]**2 + imu["az"]**2) ** 0.5

display(imu.head())
print(imu.describe())

# 3. Трансформация в аналитический колоночный формат.
PARQUET = Path("/content/robot_imu.parquet")
imu.to_parquet(PARQUET, index=False)
print("Parquet:", PARQUET, "size =", PARQUET.stat().st_size, "bytes")

# 4. Визуализация извлечённой телеметрии.
long = imu.melt(
    id_vars=["time_s"],
    value_vars=["ax", "ay", "az"],
    var_name="axis",
    value_name="acceleration"
)

fig = px.line(long, x="time_s", y="acceleration", color="axis",
              title="Декодированная ROS 2 IMU-телеметрия из MCAP")
fig.show()

# 5. Проверки.
assert len(imu) == 1000
assert {"ax", "ay", "az"}.issubset(imu.columns)
assert imu["ax"].abs().max() > 2.5

# Дополнительно: rosbags 0.11.5 используется для чтения rosbag2 SQLite/MCAP
# из файлов, полученных от реального ROS 2. Полноценная установка ROS не требуется.
import rosbags
print("rosbags imported successfully")


## TEACH CARD

После выполнения кода заполните `TEACH CARD` из `.md`-файла лабораторной работы и приложите его к отчёту.
